# Python Foundations for Data Engineering: Full Overview
### Instructor Demo (Charter Oak Mutual claims) | Python 3.13

This notebook covers every concept the drills practice, from the **basics** to
**modern Python 3.7+ features**, organized the way a data engineer works:

> **Ingest -> Inspect -> Transform -> Analyze -> Output**

Teach it top to bottom in VS Code, then move to `demo_pipeline.py` to show the
script form your drills and homework use.

**Map of this notebook**
1. Basics: types, strings, collections, control flow, functions, exceptions
2. Data I/O: text, CSV, JSON, SQLite
3. Modern Python (3.7+): comprehensions, lambdas/map/filter, generators, walrus,
   match/case, classes, dataclasses + type hints
4. Put it together: a small pipeline, then notebook -> script

# Part 1: The Basics

## Variables and types
Everything is an object with a type. Know your types.

In [ ]:
claim_id = "CLM-D01"   # str
reserve = 5000.0        # float
open_count = 3          # int
is_open = True          # bool
adjuster = None         # NoneType (absence of a value)

for value in (claim_id, reserve, open_count, is_open, adjuster):
    print(f"{type(value).__name__:<8} -> {value}")

## f-strings and format specifiers
Format numbers for humans.

In [ ]:
reserve = 5000.0
claims = 3421
rate = 0.2012
print(f"${reserve:,.2f} reserve across {claims:,} claims")
print(f"no-show rate: {rate:.1%}")   # percent format

## Strings
Text cleaning is half of data engineering.

In [ ]:
raw = "  Auto, Property , Liability  "
parts = [p.strip().lower() for p in raw.split(",")]
print(parts)
print("auto".upper(), "PROPERTY".lower(), "liability".capitalize())
print("CLM-D01".startswith("CLM"), "CLM-D01"[-2:])

## Collections: list, tuple, set, dict
The four containers you reach for daily.

In [ ]:
# list: ordered, mutable
queue = ["CLM-1", "CLM-2", "CLM-3"]
queue.append("CLM-4")
print("list:", queue, "| first two:", queue[:2], "| len:", len(queue))

# tuple: ordered, immutable, great for fixed records and unpacking
record = ("CLM-1", "auto", 5000.0)
cid, ptype, res = record
print("tuple unpack:", cid, ptype, res)

# set: unique values, fast membership
types = {"auto", "property", "auto", "liability"}
print("set:", sorted(types), "| 'auto' in set:", "auto" in types)

# dict: key -> value
claim = {"claim_id": "CLM-1", "reserve": 5000.0}
claim["paid"] = 1200.0
print("dict.get with default:", claim.get("status", "open"))
for k, v in claim.items():
    print("   ", k, "=", v)

## Control flow: conditionals
`if / elif / else` with comparison and logical operators.

In [ ]:
reserve, manager_override = 22000, False
if reserve >= 20000 and not manager_override:
    tier = "severe"
elif reserve >= 5000:
    tier = "major"
else:
    tier = "minor"
print("tier:", tier)

## Control flow: loops
`for`, `range`, `enumerate`, `zip`, and `while`.

In [ ]:
amounts = [1200, 5000, 800]
for i, amount in enumerate(amounts, start=1):
    print(f"row {i}: {amount}")

for ptype, amount in zip(["auto", "property"], [1200, 5000]):
    print(ptype, amount)

n = 3
while n > 0:
    print("retry", n)
    n -= 1

## Functions
Name a calculation once; default arguments and docstrings.

In [ ]:
def loss_ratio(paid, reserve, round_to=1):
    """Paid divided by reserve, as a percentage."""
    if reserve == 0:
        return 0.0
    return round(paid / reserve * 100, round_to)

print(loss_ratio(1200, 5000), loss_ratio(0, 0), loss_ratio(1234, 5000, round_to=3))

## Exceptions
Real data is messy. Handle bad values instead of crashing.

In [ ]:
def to_float(raw):
    try:
        return float(raw)
    except ValueError:
        return None
    finally:
        pass  # finally always runs; used for cleanup

print(to_float("18.5"), to_float("N/A"))

# Part 2: Data I/O (ingest and output)
Data engineering starts by reading data and ends by writing it.

## Text files

In [ ]:
from pathlib import Path
Path("demo_note.txt").write_text("claims processed: 8\n")
print(Path("demo_note.txt").read_text().strip())
Path("demo_note.txt").unlink()  # cleanup

## CSV: ingest the claims (we reuse `claims` for the rest of the notebook)

In [ ]:
import csv
claims = []
with open("claims.csv", newline="") as f:
    for row in csv.DictReader(f):
        row["reserve"] = float(row["reserve"])
        row["paid"] = float(row["paid"])
        claims.append(row)
print(f"ingested {len(claims)} claims; first:", claims[0])

## JSON
The shape a REST API returns (Day 3). Serialize and parse.

In [ ]:
import json
payload = json.dumps(claims[0], indent=2)
print(payload)
parsed = json.loads(payload)
print("round-tripped:", parsed["claim_id"])

## SQLite
A real database with SQL, no install. Bridge to the SQL weeks.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")   # in-memory db for the demo
cur = conn.cursor()
cur.execute("CREATE TABLE claims (id TEXT, ptype TEXT, reserve REAL)")
cur.executemany(
    "INSERT INTO claims VALUES (?, ?, ?)",
    [(c["claim_id"], c["policy_type"], c["reserve"]) for c in claims],
)
conn.commit()
for row in cur.execute("SELECT ptype, SUM(reserve) FROM claims GROUP BY ptype ORDER BY ptype"):
    print(row)
conn.close()

# Part 3: Modern Python (3.7+)
The features that separate tutorial code from professional code.

## Comprehensions
Build a list, dict, or set in one readable line.

In [ ]:
open_ids = [c["claim_id"] for c in claims if c["status"] == "open"]
reserve_by_id = {c["claim_id"]: c["reserve"] for c in claims}
policy_types = {c["policy_type"] for c in claims}
print(open_ids)
print(sorted(policy_types))

## Lambdas, map, filter
The transform pattern you reuse in PySpark (Week 5).

In [ ]:
by_reserve = sorted(claims, key=lambda c: c["reserve"], reverse=True)
print("top 3 by reserve:", [c["claim_id"] for c in by_reserve][:3])
bumped = list(map(lambda c: round(c["reserve"] * 1.10, 2), claims))
open_only = list(filter(lambda c: c["status"] == "open", claims))
print("bumped:", bumped[:3], "| open:", len(open_only))

## Generators
Produce values lazily, one at a time (memory friendly for big data).

In [ ]:
def claims_over(rows, threshold):
    for c in rows:
        if c["reserve"] > threshold:
            yield c["claim_id"]

print("reserve over 10k:", list(claims_over(claims, 10000)))

## Walrus operator `:=` (3.8)
Assign and test in one expression.

In [ ]:
running = 0
for c in claims:
    if (running := running + c["paid"]) > 30000:
        print(f"cumulative paid crossed 30k at {c['claim_id']} (= {running})")
        break

## match/case (3.10)
Match the shape of data, not just one value.

In [ ]:
def route(claim):
    match claim:
        case {"status": "denied"}:
            return "closed"
        case {"policy_type": "auto", "reserve": r} if r > 4000:
            return "auto-review"
        case {"status": "open"}:
            return "standard"
        case _:
            return "manual"

for c in claims[:4]:
    print(c["claim_id"], "->", route(c))

## Classes
Bundle data and behavior. Use a class for a thing with state; a function for a pure calculation.

In [ ]:
class Claim:
    def __init__(self, claim_id, reserve, paid):
        self.claim_id = claim_id
        self.reserve = reserve
        self.paid = paid

    def burn(self):
        return round(self.paid / self.reserve * 100, 1)

c = Claim("CLM-1", 5000.0, 1200.0)
print(c.claim_id, "burn:", c.burn(), "%")

## Dataclasses (3.7) + type hints
Same class, less boilerplate. The dataclass writes `__init__`, `__repr__`, and `__eq__`.

In [ ]:
from dataclasses import dataclass

@dataclass
class ClaimDC:
    claim_id: str
    reserve: float
    paid: float = 0.0          # typed field with a default

    def burn(self) -> float:
        return round(self.paid / self.reserve * 100, 1)

dc = ClaimDC("CLM-1", 5000.0, 1200.0)
print(dc)                      # auto __repr__
print("burn:", dc.burn(), "| equal to twin:", dc == ClaimDC("CLM-1", 5000.0, 1200.0))

# Part 4: Put it together, then move to a script

## A small pipeline: aggregate and report by policy type

In [ ]:
summary = {}
for c in claims:
    bucket = summary.setdefault(c["policy_type"], {"reserve": 0.0, "paid": 0.0})
    bucket["reserve"] += c["reserve"]
    bucket["paid"] += c["paid"]

print("Loss ratio by policy type:")
for ptype in sorted(summary):
    b = summary[ptype]
    print(f"  {ptype}: {loss_ratio(b['paid'], b['reserve'])}%")

## From notebook to script: why `.py` for your homework

Notebooks are for **exploring** cell by cell. Pipelines ship as `.py` files: they
run top to bottom, version-control cleanly, and are how your drills and homework
are structured. The same logic above lives in `demo_pipeline.py` in this folder,
wrapped in functions and a `main()`.

## How to run a `.py` file

From the folder that contains the script and its data:

```bash
uv run python demo_pipeline.py
```

`uv run` uses this project's `.venv`. The workflow you will use all course:

1. Work under `student-work/week2/day2/` so a `git pull` never conflicts.
2. The `.venv` lives in that day folder; run `uv` commands from there, and select
   it as your VS Code interpreter and notebook kernel.
3. Copy a drill's `Unsolved` scaffold and any `Resources/` into your project, then
   write the logic yourself.

Now open `Python Drills/` and work them as scripts, then the mini-capstones.